# 00 — Data Audit
Replicates §2 of the Morocco BTP Nowcasting Implementation Plan.
Descriptive statistics, variable inventory, and correlation analysis.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from lamiaty.utils.logging import setup_logging
from lamiaty.config import load_settings
from lamiaty.data.loader import load_base_btp

setup_logging()
settings = load_settings("../configs", project_root="..")
df_raw = load_base_btp(settings.paths.base_btp_path)
print("Shape:", df_raw.shape)
df_raw.head()

## §2.2 Variable inventory

In [ ]:
df_raw.dtypes.to_frame("dtype")

## §2.3 Descriptive statistics

In [ ]:
df_raw.describe().T.round(1)

## §2.4 Missing values

In [ ]:
df_raw.isnull().sum().to_frame("n_missing")

## §2.4 Correlations with VA CONSTRUCTION (y-o-y)

In [ ]:
import numpy as np
from lamiaty.data.corrections import apply_all_corrections
from lamiaty.data.transforms import yoy_log_diff, assign_quarterly_to_month_end
from lamiaty.visualization.diagnostics import plot_correlation_matrix

df_corr = apply_all_corrections(df_raw, settings.corrections)

# yoy for monthly series
for col in ["consommation_ciment", "credits_equipement", "credits_immobilier",
            "lafarge_index", "investissement_etat"]:
    if col in df_corr.columns:
        df_corr[col] = yoy_log_diff(df_corr[col])

# quarter-end assignment for quarterly series
for col in ["va_construction", "ipai", "creation_emploi"]:
    if col in df_corr.columns:
        df_corr[col] = assign_quarterly_to_month_end(df_corr[col])

fig = plot_correlation_matrix(df_corr, target_col="va_construction")
fig.savefig("../docs/correlations.png", dpi=150, bbox_inches="tight")

## §2.4 Series panel overview

In [ ]:
from lamiaty.visualization.diagnostics import plot_series_panel
fig = plot_series_panel(df_corr, title="Base BTP — all series (yoy transformed)")
fig.savefig("../docs/series_panel.png", dpi=150, bbox_inches="tight")